# **Initialization**

In [1]:
print('Start')

Start


In [2]:
import glob
import random
import os
import time
import pandas as pd
import modified_didppy as m_dp

# **Data**

In [5]:
def read_tsp_cappart_format(file_path):
    """
    Parses the TSP/TSPTW text files from the specified directory.
    Structure:
    - n (int)
    - n*n distance matrix entries
    - n*2 time window entries (ignored for TSP)
    - n x_coords (ignored)
    - n y_coords (ignored)
    """
    with open(file_path, 'r') as f:
        # split() handles all whitespace (newlines and spaces) automatically
        values = f.read().split()

    iterator = iter(values)
    
    try:
        # 1. Read Number of Nodes
        n = int(next(iterator))
        
        # 2. Read Distance Matrix (n x n)
        # The file contains a flattened list of integer distances
        c = []
        for i in range(n):
            row = []
            for j in range(n):
                val = float(next(iterator)) # Read as float first to be safe
                row.append(int(val))        # Convert to int as per your DIDP model type
            c.append(row)
            
        # The rest of the file (Time windows, coords) is ignored for pure TSP
        # but the iterator ensures we consumed exactly what we needed.
        
        return n, c

    except StopIteration:
        raise ValueError(f"File {file_path} ended unexpectedly.")

# **Original DIDP model with single dual bounds**

In [6]:
# ==========================================
# 1. Configuration & File Selection
# ==========================================

# Directory containing the instances
folder_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_TSPTW_dual_bounds_and_models\n20"

# Get all .txt files
all_files = glob.glob(os.path.join(folder_path, "*.txt"))

# Select 20 random instances (or all if less than 20)
num_instances_to_test = 20
if len(all_files) > num_instances_to_test:
    selected_files = random.sample(all_files, num_instances_to_test)
else:
    selected_files = all_files

print(f"Found {len(all_files)} files. Selected {len(selected_files)} for testing.")
print("Selected Instances:")
for f in selected_files:
    print(f" - {os.path.basename(f)}")
print("-" * 50)

# ==========================================
# 2. Testing Loop
# ==========================================

results_data = []
output_csv_name = "TSP_single_dual_bound_results.csv"

for i, file_path in enumerate(selected_files):
    instance_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(selected_files)}] Processing: {instance_name}")
    
    try:
        # --- A. Read Data ---
        # Using the function you defined in previous cells
        n, c = read_tsp_cappart_format(file_path)
        
        # --- B. Initialize Model (MUST be done fresh for every instance) ---
        model = m_dp.Model(maximize=False, float_cost=False)
        customer = model.add_object_type(number=n)
        
        # State Variables
        unvisited = model.add_set_var(object_type=customer, target=list(range(1, n)))
        location = model.add_element_var(object_type=customer, target=0)
        
        # Resource Tables
        travel_time = model.add_int_table(c)
        
        # Transitions: Visit customer j
        for j in range(1, n):
            visit = m_dp.Transition(
                name="visit {}".format(j),
                cost=travel_time[location, j] + m_dp.IntExpr.state_cost(),
                preconditions=[unvisited.contains(j)],
                effects=[
                    (unvisited, unvisited.remove(j)),
                    (location, j),
                ],
            )
            model.add_transition(visit)
        
        # Transitions: Return to depot
        return_to_depot = m_dp.Transition(
            name="return",
            cost=travel_time[location, 0] + m_dp.IntExpr.state_cost(),
            effects=[(location, 0)],
            preconditions=[unvisited.is_empty(), location != 0],
        )
        model.add_transition(return_to_depot)
        
        # Base Case
        model.add_base_case([unvisited.is_empty(), location == 0])
        
        # --- C. Dual Bounds ---
        # Min outgoing edge
        min_to = model.add_int_table(
            [min(c[k][j] for k in range(n) if k != j) for j in range(n)]
        )
        model.add_dual_bound(min_to[unvisited] + (location != 0).if_then_else(min_to[0], 0))
        
        # Min incoming edge
        min_from = model.add_int_table(
            [min(c[j][k] for k in range(n) if k != j) for j in range(n)]
        )
        model.add_dual_bound(
            min_from[unvisited] + (location != 0).if_then_else(min_from[location], 0)
        )

        # --- D. Solver Execution ---
        t_start = time.time()
        
        # Solver with 1 hour limit (3600 seconds)
        solver = m_dp.CABS(
            model,
            quiet=True, # Set to True to reduce console clutter during batch processing
            time_limit=3600
        )
        
        solution = solver.search()
        
        t_end = time.time()
        duration = t_end - t_start
        
        # --- E. Logging ---
        cost = solution.cost
        is_optimal = solution.is_optimal
        nodes_gen = solution.generated
        nodes_exp = solution.expanded
        
        print(f"   -> Done. Cost: {cost}, Time: {duration:.2f}s, Optimal: {is_optimal}")

        # Append to results
        results_data.append({
            "Instance": instance_name,
            "Cost": cost,
            "Nodes Expanded": nodes_exp,
            "Nodes Generated": nodes_gen,
            "Running Time (s)": duration,
            "Is Optimal": is_optimal
        })

    except Exception as e:
        print(f"   -> ERROR processing {instance_name}: {e}")
        results_data.append({
            "Instance": instance_name,
            "Cost": "Error",
            "Nodes Expanded": 0,
            "Nodes Generated": 0,
            "Running Time (s)": 0,
            "Is Optimal": "Error"
        })

    # --- F. Intermediate Save ---
    # Save the dataframe after every loop so you don't lose progress if it crashes
    df_results = pd.DataFrame(results_data)
    df_results.to_csv(output_csv_name, index=False)

print("\n" + "="*50)
print("Batch Testing Complete.")
print(f"Results saved to {output_csv_name}")
print(df_results)

Found 100 files. Selected 20 for testing.
Selected Instances:
 - 98.txt
 - 91.txt
 - 18.txt
 - 12.txt
 - 49.txt
 - 99.txt
 - 85.txt
 - 26.txt
 - 1.txt
 - 53.txt
 - 86.txt
 - 95.txt
 - 19.txt
 - 21.txt
 - 84.txt
 - 29.txt
 - 47.txt
 - 71.txt
 - 5.txt
 - 30.txt
--------------------------------------------------

[1/20] Processing: 98.txt
   -> Done. Cost: 356, Time: 52.76s, Optimal: True

[2/20] Processing: 91.txt
   -> Done. Cost: 365, Time: 154.55s, Optimal: True

[3/20] Processing: 18.txt
   -> Done. Cost: 384, Time: 103.07s, Optimal: True

[4/20] Processing: 12.txt
   -> Done. Cost: 399, Time: 170.04s, Optimal: True

[5/20] Processing: 49.txt
   -> Done. Cost: 360, Time: 24.67s, Optimal: True

[6/20] Processing: 99.txt
   -> Done. Cost: 322, Time: 178.73s, Optimal: True

[7/20] Processing: 85.txt
   -> Done. Cost: 371, Time: 89.13s, Optimal: True

[8/20] Processing: 26.txt
   -> Done. Cost: 389, Time: 209.19s, Optimal: True

[9/20] Processing: 1.txt
   -> Done. Cost: 390, Time: 90.38